In [1]:
from pyspark.sql import functions as F
from datetime import datetime

BATCH_ID = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
RAW = "Files/raw"
TABLES = ["products", "depots", "orders", "deliveries"]

for name in TABLES:
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "false")   # bronze preserves the source verbatim
          .csv(f"{RAW}/{name}.csv"))

    df = (df
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(f"{name}.csv"))
          .withColumn("_ingest_batch", F.lit(BATCH_ID)))

    (df.write
       .mode("overwrite")
       .format("delta")
       .saveAsTable(f"bronze_{name}"))

    print(f"bronze_{name:<12} {df.count():>7,} rows")

StatementMeta(, beae132a-e576-4119-8d1f-d5fa7f30d6a0, 3, Finished, Available, Finished, False)

bronze_products          52 rows
bronze_depots            12 rows
bronze_orders        17,912 rows
bronze_deliveries     5,546 rows


In [2]:
for name in TABLES:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM bronze_{name}").collect()[0]["n"]
    print(f"bronze_{name:<12} {n:>7,}")

StatementMeta(, beae132a-e576-4119-8d1f-d5fa7f30d6a0, 4, Finished, Available, Finished, False)

bronze_products          52
bronze_depots            12
bronze_orders        17,912
bronze_deliveries     5,546
